# MV00 DDP-B16 — Fresh Epoch-1 GPU Acceptance

**Nonofficial and bounded to one epoch.** Uses exactly two Tesla T4 GPUs, Frozen Split B, global batch 16, image size 1024. No B-TEST. This is the final GPU checkpoint test before creating the continuous 100-epoch notebook.

Attach the seven pre-existing Kaggle datasets and the new offline source bundle. Set only the three pins in the code cell; then use **Save Version → Save & Run All**. Outputs are written into `/kaggle/working/MV00_B16_ACCEPTANCE_OUTPUT`.


In [ ]:
# MV00 bounded epoch-1 acceptance: one code cell after editing the 3 pins below.
# This is NOT the final 100-epoch production notebook.
BUNDLE_DATASET_ROOT = "/kaggle/input/datasets/REPLACE_WITH_YOUR_SOURCE_BUNDLE_DATASET"
EXPECTED_GIT_HEAD = "REPLACE_WITH_40_CHARACTER_GIT_SHA"
EXPECTED_BUNDLE_SHA256 = "REPLACE_WITH_64_CHARACTER_BUNDLE_SHA256"

import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import torch

assert re.fullmatch(r"[0-9a-f]{40}", EXPECTED_GIT_HEAD), "SET_EXACT_RELEASE_GIT_SHA"
assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_BUNDLE_SHA256), "SET_EXACT_BUNDLE_SHA"
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "REQUIRE_T4_X2"
assert all("T4" in torch.cuda.get_device_name(i).upper() for i in (0, 1)), "NOT_T4_X2"

inputs = {
    "n1": Path("/kaggle/input/datasets/awaishit/detection-frozen-git-n1-accepted-r1"),
    "handoff": Path("/kaggle/input/datasets/awaishit/detection-project-training-handoff-r1"),
    "legacy": Path("/kaggle/input/datasets/awaishit/detection-legacy-notebook-files-flat-r1"),
    "split_b": Path("/kaggle/input/datasets/utopianstar/grazpedwri-dx-split-b"),
    "assets": Path("/kaggle/input/datasets/nithoukhan/d4e-r12-accepted-assets"),
    "archives": Path("/kaggle/input/datasets/nithoukhan/d4e-r12-raw-archives-v1"),
    "evidence": Path("/kaggle/input/datasets/nithoukhan2/d4f-r7d-legacy-exact-source-evidence-r1"),
}
assert all(root.is_dir() for root in inputs.values()), "MOUNT_ALL_EXISTING_7_DATASETS"
bundle = Path(BUNDLE_DATASET_ROOT) / "RESEMA_DDP_B16_ACCEPTANCE.bundle"
assert bundle.is_file() and not bundle.is_symlink(), "MOUNT_NEW_SOURCE_BUNDLE_DATASET"

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(bundle) == EXPECTED_BUNDLE_SHA256, "OFFLINE_SOURCE_BUNDLE_PIN_MISMATCH"

work = Path("/kaggle/working/MV00_B16_ACCEPTANCE_PREP")
repo = work / "repo"
stage = Path("/kaggle/working/MV00_B16_FRESH_STAGE")
g4_root = Path("/kaggle/working/MV00_B16_FRESH_G4")
project = Path("/kaggle/working/MV00_B16_NATIVE_PROJECT")
artifacts = Path("/kaggle/working/MV00_B16_ACCEPTANCE_OUTPUT")
run_id = "MV00_B16_FRESH_ACCEPTANCE"
assert not any(p.exists() for p in (work, stage, g4_root, project, artifacts)), "FRESH_OUTPUTS_REQUIRED"

work.mkdir()
subprocess.run(["git", "clone", "--quiet", str(bundle), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--quiet", "--detach", EXPECTED_GIT_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_GIT_HEAD, "SOURCE_COMMIT_MISMATCH"
assert not subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain", "--untracked-files=no"], text=True).strip()

sys.dont_write_bytecode = True
sys.path.insert(0, str(repo))
from research.r9_training.ddp_b16.r0_relocation import restore_r0_config
from research.r9_training.ddp_b16.ddp_worker import _unique_json

configs = sorted(inputs["n1"].rglob("R0_RELOCATED_CONFIG_CANDIDATE.json"))
assert len(configs) == 1, "R0_CONFIG_MISSING_OR_AMBIGUOUS"
config = configs[0]
r0_file = (repo / "research/r9_training/frozen_source/ultralytics/models"
           "/yolo/detect/d4e_r3_read_only_identity_preflight.py")
spec = importlib.util.spec_from_file_location("_mv00_original_r0", r0_file)
r0 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(r0)

source_roots = [inputs[k] for k in ("n1", "handoff", "legacy", "assets", "archives", "evidence")]
restored = restore_r0_config(
    config, source_roots,
    asset_pins=r0.ASSET_PINS, overlay_pins=r0.OVERLAY_PINS,
    code_pins=r0.R21_PINS, native_blob_pin=r0.NATIVE_BLOB_SHA1,
    runtime_yaml_pin="f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",
    mirrored_handoff=inputs["handoff"],
    archives_roots=[inputs["assets"], inputs["archives"]],
)
r0_receipt = r0.verify_inputs(
    _unique_json(config), "MV00", run_id,
    "f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",
    str(repo / "research/r9_training/R9_RELEASE_RECIPE_CANDIDATE_R1.json"),
)
assert r0_receipt["status"] == "PASS_LOCAL_FILES_MATCH_PINNED_BYTES_NOT_GPU_ACCEPTANCE"
assert r0_receipt["b_test_access"] == "NONE"
print("R0 ASSETS VERIFIED; FILES RELOCATED:", restored["restored"], flush=True)

plan = {
    "schema": "VGRA_DDP_B16_WORKER_PLAN_REVIEW_V1",
    "arm": "MV00",
    "run_id": run_id,
    "mode": "FRESH_FROM_EARLY",
    "checkout_root": str(repo),
    "expected_git_head": head,
    "r0_input_config": str(config),
    "stage_root": str(stage),
    "b_test_access": "NONE",
}
plan_path = work / "plan.json"
plan_path.write_text(json.dumps(plan, indent=2, sort_keys=True) + "\n", encoding="utf-8")
env = dict(os.environ)
env.update({
    "CUDA_VISIBLE_DEVICES": "0,1",
    "OMP_NUM_THREADS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONPATH": str(repo) + os.pathsep + env.get("PYTHONPATH", ""),
})
command = [
    sys.executable, "-m",
    "research.r9_training.ddp_b16.ddp_gpu_epoch_acceptance",
    "--plan", str(plan_path), "--project", str(project),
    "--g4-root", str(g4_root), "--artifacts", str(artifacts),
    "--accept-one-epoch-gpu",
]
print("RUNNING: MV00 FRESH NONOFFICIAL EPOCH-1 ACCEPTANCE", flush=True)
print("GIT:", head, "| T4x2 | global batch 16 | 1024 | B-TEST SEALED", flush=True)
result = subprocess.run(command, cwd=repo, env=env, check=False)
assert result.returncode == 0, "ACCEPTANCE_FAILED_INSPECT_" + str(artifacts / "TWO_RANK_EPOCH1.log")
receipt = json.loads((artifacts / "g4" / "ACCEPTANCE.json").read_text("utf-8"))
assert receipt["status"] == "NONOFFICIAL_EPOCH1_G4_SAVED_NOT_OFFSESSION_VERIFIED"
print("MV00 EPOCH1 G4 CANDIDATE COMPLETE")
print("SAVED OUTPUT:", artifacts)
print("EXTERNAL MANIFEST SHA256:", receipt["manifest_sha256"])
print("SAVE VERSION OUTPUT AND VERIFY IN A SEPARATE SESSION BEFORE OFFICIAL TRAINING")
